<a href="https://colab.research.google.com/github/vishaljoshi24/DungeonsAndDragonsTurnClassification/blob/main/zero_R_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone 'https://github.com/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/'

In [ ]:
# !pip install dspy==3.1.0
!pip install dspy==3.2.1

In [ ]:
import pandas as pd
import dspy

In [ ]:
testing_df = pd.read_excel()

In [ ]:
test_set = []

for context, current_turn, category in testing_df.values:
    examples = dspy.Example(context=context, turn=current_turn, category=category).with_inputs("context", "turn")
    test_set.append(examples)

In [ ]:
test_set

In [ ]:
lm = dspy.LM('ollama_chat/qwen3:8b', api_base = 'http://localhost:11434', api_key='', max_tokens=4096)
dspy.configure(lm=lm)

In [ ]:
class BaselineClassifier(dspy.Signature):
    """Given the context for Dungeons & Dragons game turn, the game turn itself and the majority class label, label the dataset using the majority class label"""
    context: str = dspy.InputField(desc = "The three previous game turns which describe a player's action or their dialogue.")
    turn: str = dspy.InputField(desc="The current turn taken by a player, which can include a description of an action or a piece of dialogue.")
    mode: str = dspy.InputField(desc = "The class which the majority of the game turns are labelled with.")
    zeroRclass: str = dspy.OutputField(desc = "The label for the game turn.")

In [ ]:
class ZeroRClassification(dspy.Module):
  def __init__(self):
    self.zeroRclassifier = dspy.Predict(BaselineClassifier, caching=False)

  def forward(self, context, turn, mode, **kwargs):
    predicted_zeroR_class = self.zeroRclassifier(context=context, turn=turn, mode=mode)
    return predicted_zeroR_class

In [ ]:
zeroRclassify = ZeroRClassification()
def zeroR_classify_turn(context, turn, mode):
    try:
        predicted_class = zeroRclassify(context=context, turn=turn, mode=mode)
        return predicted_class
    except Exception as e:
        return 0

In [ ]:
baseline_predictions = []

for i in range(len(test_set[0:10])):
    baseline_predictions.append(zeroR_classify_turn(test_set[i]['context'], test_set[i]['turn'], ''))


In [ ]:
baseline_predictions

In [ ]:
predicted_categories = []

for i in range(len(baseline_predictions)):
  baseline_categories.append(baseline_predictions[i].zeroRclass)

In [ ]:
baseline_categories

In [ ]:
true_categories = []

for i in range(len(trainset[0:10])):
  true_categories.append(test_set[i]['category'])

In [ ]:
true_categories

In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

def f1(true_categories, predicted_categories, trace=None):
  for i in range(len(true_categories)):
    return f1_score(true_categories, predicted_categories, average='weighted')

def precision(true_categories, predicted_categories, trace=None):
  for i in range(len(true_categories)):
    return precision_score(true_categories, predicted_categories, labels=label_list, average=None)

def recall(true_categories, predicted_categories, trace=None):
  for i in range(len(true_categories)):
    return recall_score(true_categories, predicted_categories, labels = label_list, average=None)

In [ ]:
recall_list = []
precision_list = []

for i in range(len(true_categories)):
  precision_list.append(precision(true_categories, baseline_categories))

for i in range(len(true_categories)):
  recall_list.append(recall(true_categories, baseline_categories))

In [ ]:
precision_list

In [ ]:
recall_list

In [ ]:
f1(true_categories, predicted_categories)